# Set of simulations for DPS

Validation notebook for simulated dataset related to technical note PLATO-PL-DLR-TN-0108.

Created: 2025-03-10 - Nicholas Jannsen

Updated: 2025-05-30 - Nicholas Jannsen 
```
commit 72e4f639fa20987cda3e4a15b51cbc6c9f16b997
```

### Setup notebook

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib notebook

### Imports

In [ ]:
import os
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import colors
from matplotlib.colors import Normalize, LogNorm
from scipy import ndimage
from astropy import units as u
from astropy.coordinates import SkyCoord
from astroquery.simbad import Simbad

# PlatoSim
import platosim.plot       as pt
import platosim.utilities  as ut
import platosim.referenceFrames as rf
from platosim.h5           import h5get, h5ls
from platosim.simfile      import SimFile
from platosim.simulation   import Simulation
from platosim.utilities    import imageNorm
from platosim.slurm        import workerOverview
from platosim.matplotlibrc import setup_notebook
setup_notebook()

from IPython.display import display, HTML
display(HTML("<style>.container {width:80% !important; }</style>"))

### Internal directories

In [ ]:
path = os.getenv('PLATO_WORKDIR') + 'DLR108'
idir = f'{path}/input'
odir = f'{path}/test_data'
fdir = f'{path}/plots'

---
## Bright stars not found by Gaia
---

In [ ]:
# Setup simbad query
simbad = Simbad()
simbad.ROW_LIMIT = 10000
simbad.timeout   = 2000 

# Perform query
table_gaia = simbad.query_criteria("Vmag < 5 & Gmag < 6")
table_none = simbad.query_criteria("Vmag < 5")

# Transform astropy table to pandas
df_gaia = table_gaia.to_pandas()
df_none = table_none.to_pandas()

In [ ]:
# Check how many stars that are not in Gaia
df_none[~df_none['MAIN_ID'].isin(df_gaia['MAIN_ID'])]

---
## Stellar input catalogues
---

In [ ]:
df1 = pd.read_feather(f"{idir}/starcat_GaiaDR3_LOPS2_group1.ftr")
df2 = pd.read_feather(f"{idir}/starcat_GaiaDR3_LOPS2_group2.ftr")
df3 = pd.read_feather(f"{idir}/starcat_GaiaDR3_LOPS2_group3.ftr")
df4 = pd.read_feather(f"{idir}/starcat_GaiaDR3_LOPS2_group4.ftr")
df = pd.concat([df1, df2, df3, df4])
df = df.drop_duplicates()
df = df.reset_index(drop=True)

In [ ]:
# Show star catalogues 
df0 = df.iloc[::100]
fig, ax = pt.plotPlatoFOV('LOPS2', raStars=df0.ra, decStars=df0.dec, magStars=None, 
                          system="icrs", showGroups=True, ncamStars=False, fovSize=35, 
                          fs=20, ms=1, aa=0.5, figsize=(9,9))

---
## N-CAM simulations
---

In [ ]:
group, cam = 4, 1

### Load test data

In [ ]:
f1 = SimFile(f'{odir}/Ncam{group}.{cam}_Q1_ccd1.hdf5')
f2 = SimFile(f'{odir}/Ncam{group}.{cam}_Q1_ccd2.hdf5')
f3 = SimFile(f'{odir}/Ncam{group}.{cam}_Q1_ccd3.hdf5')
f4 = SimFile(f'{odir}/Ncam{group}.{cam}_Q1_ccd4.hdf5')

df1 = pd.read_feather(f"{odir}/Ncam{group}.{cam}_Q1_ccd1.ftr")
df2 = pd.read_feather(f"{odir}/Ncam{group}.{cam}_Q1_ccd2.ftr")
df3 = pd.read_feather(f"{odir}/Ncam{group}.{cam}_Q1_ccd3.ftr")
df4 = pd.read_feather(f"{odir}/Ncam{group}.{cam}_Q1_ccd4.ftr")

im1 = f1.getImage(0)
im2 = f2.getImage(0)
im3 = f3.getImage(0)
im4 = f4.getImage(0)

df1.head()

### Show focal plane

In [ ]:
# Plot FPA in a subfplot
fig = plt.figure(figsize=(9,9))
ax1 = fig.add_subplot(221)
ax2 = fig.add_subplot(222)
ax3 = fig.add_subplot(223)
ax4 = fig.add_subplot(224)

axs = [ax1, ax2, ax3, ax4]
ims = [im1, im4, im2, im3]
ang = [180,  270,  90,  0]

for ax, im, a in zip(axs, ims, ang):
    
    im = imageNorm(im, "log", sigma=0.5)
    vmin = im.min()
    vmax = im.max()
    norm = None

    ax.imshow(ndimage.rotate(im, a), norm=norm, origin='lower')
    ax.xaxis.set_ticks([])
    ax.yaxis.set_ticks([])
    
plt.tight_layout(pad=0)
# fig.savefig('FPAgroup4.png', bbox_inches='tight', dpi=200)

### Zoom-in on LMC

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(9,9))
im = imageNorm(im1, "log", sigma=0.5)
ax.imshow(ndimage.rotate(im, 180), norm=None, cmap='cubehelix', origin='lower')
ax.set_xlim(500, 2300)
ax.set_ylim(250, 1800)
ax.set_xlabel(r'Column pixel, $i$')
ax.set_ylabel(r'Row pixel, $j$')
plt.tight_layout()
# fig.savefig('LMC_zoom.png', bbox_inches='tight', dpi=200)

In [ ]:
# Plot for thesis cover
fig, ax = plt.subplots(1, 1, figsize=(18,18))
im = imageNorm(im1, "log", sigma=0.5)
ax.imshow(ndimage.rotate(im, 180), norm=None, cmap='cubehelix', origin='lower')
plt.tight_layout()
fig.savefig(f'{fdir}/fullframe.png', bbox_inches='tight', dpi=300)

### Check star positions at the FOV edge

In [ ]:
im, df = im2, df2
img = ndimage.rotate(imageNorm(im, "log", sigma=0.5), 0)
box = [3400, 3500, 1180, 1280]

fig, ax = plt.subplots(1, 2, figsize=(9,5))

# Plot image
ax[0].imshow(img, norm=None, cmap='cubehelix', origin='lower')
ax[0].plot(df.xCCD-0.5, df.yCCD-0.5, 'r.', ms=0.05)
ax[0].plot([box[0], box[0]], [box[2], box[3]], 'b-', lw=2)
ax[0].plot([box[1], box[1]], [box[2], box[3]], 'b-', lw=2)
ax[0].plot([box[0], box[1]], [box[2], box[2]], 'b-', lw=2)
ax[0].plot([box[0], box[1]], [box[3], box[3]], 'b-', lw=2)
ax[0].set_xlabel(r'Column pixel, $i$')
ax[0].set_ylabel(r'Row pixel, $j$')
ax[0].set_xlim(0, 4510)
ax[0].set_ylim(0, 4510)

# Zoom-in on edge
ax[1].imshow(img, norm=None, cmap='cubehelix', origin='lower')
ax[1].plot(df.xCCD, df.yCCD, 'r.')
ax[1].set_xlim(box[0], box[1])
ax[1].set_ylim(box[2], box[3])
ax[1].set_xlabel(r'Column pixel, $i$')
ax[1].set_ylabel(r'Row pixel, $j$')

plt.tight_layout()
# fig.savefig('LMC_fullframe.png', bbox_inches='tight', dpi=200)

### Check camera pointings

In [ ]:
# Fetch pointing of camera groups. Below we compare to camera pointing reported by Juan Cabrera:
alpha, delta, kappa = ut.getPointingField('LOPS2')
raGroups, decGroups = rf.getCameraGroupCoordinates(alpha, delta, kappa)
df = pd.DataFrame()
df['group'] = [1,2,3,4]
df['RA_PlatoSim']  = np.rad2deg(raGroups)
df['Dec_PlatoSim'] = np.rad2deg(decGroups)
df['RA_Juan']  = [108.15774225, 101.46602301, 84.57884632, 86.94866734]
df['Dec_Juan'] = [-51.95332635, -39.82043331, -42.61367055, -55.50513]
df

In [ ]:
s = 50
x    = np.concatenate([df1.ra.iloc[::s], df2.ra.iloc[::s], df3.ra.iloc[::s], df4.ra.iloc[::s]])
y    = np.concatenate([df1.dec.iloc[::s], df2.dec.iloc[::s], df3.dec.iloc[::s], df4.dec.iloc[::s]])
mag  = np.concatenate([df1.Pmag.iloc[::s], df2.Pmag.iloc[::s], df3.Pmag.iloc[::s], df4.Pmag.iloc[::s]])
equa = SkyCoord(x, y, frame='icrs', unit=u.deg)

In [ ]:
# Show stars detected on the CCDs in an aitoff sky projection 
fig, ax = pt.plotPlatoFOV('LOPS2', raStars=x, decStars=y, magStars=None, system="icrs",
                          showGroups=True, ncamStars=False, fovSize=35, fs=20, aa=0.1, figsize=(9,9))

# Plot a comparison in the pointing between PlatoSim and Juan's coordinates
camPointing = SkyCoord(df.RA_Juan*u.deg, df.Dec_Juan*u.deg, frame='icrs', unit='deg')
for i, c in zip(range(4), ['b', 'limegreen', 'yellow', 'r']):
    ax.plot(camPointing[i].ra.deg, camPointing[i].dec.deg, 'o', ms=5, color=c,
            mec='k', transform=ax.get_transform('world'), zorder=6, label=f'Group {i+1}')

# Plot LMC pointing
lmc = SkyCoord(80.8917*u.deg, -69.7511*u.deg, frame='icrs', unit='deg')
ax.plot(lmc.ra.deg, lmc.dec.deg, '*', ms=10, color='c', transform=ax.get_transform('world'), zorder=6, label='LMC');

---
## F-CAM simulations
---

In [ ]:
# Choose filter: blue=1 and red=2
cam = 2
odir = os.getenv('PLATO_WORKDIR') + '/DLR108/dataFCAM/maps'

### Load test data

In [ ]:
f1 = SimFile(f'{odir}/Fcam{cam}_Q1_ccd1.hdf5')
f2 = SimFile(f'{odir}/Fcam{cam}_Q1_ccd2.hdf5')
f3 = SimFile(f'{odir}/Fcam{cam}_Q1_ccd3.hdf5')
f4 = SimFile(f'{odir}/Fcam{cam}_Q1_ccd4.hdf5')

df1 = pd.read_feather(f"{odir}/Fcam{cam}_Q1_ccd1.ftr")
df2 = pd.read_feather(f"{odir}/Fcam{cam}_Q1_ccd2.ftr")
df3 = pd.read_feather(f"{odir}/Fcam{cam}_Q1_ccd3.ftr")
df4 = pd.read_feather(f"{odir}/Fcam{cam}_Q1_ccd4.ftr")

im1 = f1.getImage(0)
im2 = f2.getImage(0)
im3 = f3.getImage(0)
im4 = f4.getImage(0)

df1.head()

### Show focal plane

In [ ]:
# Plot FPA in a subplot
fig = plt.figure(figsize=(9,9))
ax1 = fig.add_subplot(221)
ax2 = fig.add_subplot(222)
ax3 = fig.add_subplot(223)
ax4 = fig.add_subplot(224)

axs = [ax1, ax2, ax3, ax4]
ims = [im1, im4, im2, im3]
ang = [180,  270,  90,  0]

for ax, im, a in zip(axs, ims, ang):
    
    im = imageNorm(im, "log", sigma=0.2)
    vmin = im.min()
    vmax = im.max()
    norm = None

    ax.imshow(ndimage.rotate(im, a), norm=norm, origin='lower')
    ax.xaxis.set_ticks([])
    ax.yaxis.set_ticks([])
    
plt.tight_layout(pad=0)
# fig.savefig('FPAgroup4.png', bbox_inches='tight', dpi=200)

### Check camera pointings

In [ ]:
s = 100 # Plot every 100th star to speed up plots
x = np.concatenate([df1.ra.iloc[::s], df2.ra.iloc[::s], df3.ra.iloc[::s], df4.ra.iloc[::s]])
y = np.concatenate([df1.dec.iloc[::s], df2.dec.iloc[::s], df3.dec.iloc[::s], df4.dec.iloc[::s]])
mag = np.concatenate([df1.Pmag.iloc[::s], df2.Pmag.iloc[::s], df3.Pmag.iloc[::s], df4.Pmag.iloc[::s]])
equa = SkyCoord(x, y, frame='icrs', unit=u.deg)

In [ ]:
# Show stars detected on the CCDs in an aitoff sky projection 
fig, ax = pt.plotPlatoFOV('LOPS2', raStars=x, decStars=y, magStars=None, system="icrs",
                          showGroups=True, ncamStars=False, fovSize=35, fs=20, aa=0.1, figsize=(9,9))

# Plot LMC pointing
lmc = SkyCoord(80.8917*u.deg, -69.7511*u.deg, frame='icrs', unit='deg')
ax.plot(lmc.ra.deg, lmc.dec.deg, '*', ms=10, color='c', transform=ax.get_transform('world'), 
        zorder=6, label='LMC');

### Check star positions at the FOV edge

In [ ]:
im, df = im2, df2
img = ndimage.rotate(imageNorm(im, "linear", sigma=1.5), 0)
box = [4250, 4420, 250, 400]

fig, ax = plt.subplots(1, 2, figsize=(9,5))

# Plot image
ax[0].imshow(img, norm=None, cmap='cubehelix', origin='lower')
ax[0].plot(df.xCCD-0.5, df.yCCD-0.5, 'r.', ms=0.05)
ax[0].plot([box[0], box[0]], [box[2], box[3]], 'b-', lw=2)
ax[0].plot([box[1], box[1]], [box[2], box[3]], 'b-', lw=2)
ax[0].plot([box[0], box[1]], [box[2], box[2]], 'b-', lw=2)
ax[0].plot([box[0], box[1]], [box[3], box[3]], 'b-', lw=2)
ax[0].set_xlabel(r'Column pixel, $i$')
ax[0].set_ylabel(r'Row pixel, $j$')

# Zoom-in on edge
ax[1].imshow(img, norm=None, cmap='cubehelix', origin='lower')
ax[1].plot(df.xCCD, df.yCCD, 'r.', ms=1)
ax[1].set_xlim(box[0], box[1])
ax[1].set_ylim(box[2], box[3])
ax[1].set_xlabel(r'Column pixel, $i$')
ax[1].set_ylabel(r'Row pixel, $j$')

plt.tight_layout()
# fig.savefig('LMC_fullframe.png', bbox_inches='tight', dpi=200)

### Check throughput compared to PLATO-DLR-PL-TN-0113

In [ ]:
import sys
mags = np.array(['08', '09', '10', '11', '12', '13'])
n = len(mags)
flux_fcam_blue = np.zeros(n)
flux_fcam_red = np.zeros(n)
for i,m in zip(range(n), mags):
    f_blue = SimFile(f"/lhome/nicholas/software/workdir/DLR108/output/000000001/000000001_Fcam1_Q1_mag{m}.hdf5")
    f_red  = SimFile(f"/lhome/nicholas/software/workdir/DLR108/output/000000001/000000001_Fcam2_Q1_mag{m}.hdf5")
    image_blue = f_blue.getImage(0) 
    image_red  = f_red.getImage(0) 
    bias = np.ones_like(image_blue) * 1000.
    flux_fcam_blue[i] = np.sum(image_blue-bias) / (0.0186 * 2.15) / 2.1
    flux_fcam_red[i]  = np.sum(image_red-bias) / (0.0186 * 2.15) / 2.1

In [ ]:
Pmag = [8, 9, 10, 11, 12, 13] 
flux_fcam_blue_juan = [6.58e4, 2.54e4, 1.01e4, 4.03e3, 1.61e3, 6.58e2]
flux_fcam_red_juan = [7.41e4, 2.95e4, 1.17e4, 4.68e3, 1.86e3, 7.41e2]

flux_fcam_blue_platosim = [65041, 38814, 12629, 5262, 1262, 589]
plt.figure(figsize=(8,6))
plt.plot(Pmag, flux_fcam_blue_juan, 'b-')
plt.plot(Pmag, flux_fcam_blue, 'b--')
plt.plot(Pmag, flux_fcam_red_juan, 'r-')
plt.plot(Pmag, flux_fcam_red, 'r--')
plt.xlabel('PLATO passband, P')
plt.ylabel('Flux [photoelectrons/s]');

---
## Statistic of simulated data
---

In [ ]:
path = '/STER/platodata/PLATOSIM/simulations_PLATO-DLR-PL-TN-0108/sim1_Oct2023/slurm'
paramFileNCAM = f'{path}/cluster_ncam.data'
paramFileFCAM = f'{path}/cluster_fcam.data'

### N-CAM resources

In [ ]:
# Use single exposure simulations as benchmark optimisation
workerLog = f'{path}/slurmNCAM_maps/run.slurm.log55428329'
df, wt = workerOverview(workerLog, paramFileNCAM, ofile=False, plot=True, fullFrame=True)

In [ ]:
# Generate new param files to more optimized batch simulations:
# (a: t < 1h, b: 1h < t < 2h, c: t > 2h):
wt0 = wt / 3600
dexA = np.where(wt0 < 1)[0]
dexB = np.where((wt0 >= 1) & (wt0 <= 2))[0]
dexC = np.where(wt0 > 2)[0]

df = pd.read_csv(paramFileNCAM)
da = df.loc[dexA]
db = df.loc[dexB]
dc = df.loc[dexC]
da.reset_index(drop=True)
db.reset_index(drop=True)
dc.reset_index(drop=True)

# Save new params files
da.to_csv(f'{path}/slurm/dataNCAM/cluster_ncam_a.txt', index=False)
db.to_csv(f'{path}/slurm/dataNCAM/cluster_ncam_b.txt', index=False)
dc.to_csv(f'{path}/slurm/dataNCAM/cluster_ncam_c.txt', index=False)

# Number of cores to be used per batch
len(dexA), len(dexB), len(dexC)

In [ ]:
# Overview for the remaining simulations
workerLog = f'{path}/slurmNCAM_sim4/run_a.slurm.log55405525'
workerLog = f'{path}/slurmNCAM_sim4/run_b.slurm.log55405526'
workerLog = f'{path}/slurmNCAM_sim4/run_c.slurm.log55405527'
df, wt = workerOverview(workerLog, paramFileNCAM, ofile=False, plot=True, fullFrame=True)

### F-CAMs

In [ ]:
# Use single exposure simulations as benchmark optimisation
workerLog = f'{path}/slurmFCAM_maps/run.slurm.log56278180'
df, wt = workerOverview(workerLog, paramFileNCAM, ofile=False, plot=True, fullFrame=True)